# Lab 8: Implementation of RNN-Based Text Classification
**Application:** Movie Review Sentiment Analyzer (IMDB)  
**Course:** Artificial Neural Networks and Deep Learning

Run all cells from top to bottom. The last cell is an interactive analyzer that waits for your input, so run it separately.

## 0. Imports and Settings

In [ ]:
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense

np.random.seed(42)
tf.random.set_seed(42)

VOCAB_SIZE = 10000   # keep only the 10,000 most frequent words
MAX_LEN = 200        # every review is cut/padded to 200 words
EMBED_DIM = 128      # size of the word vector

## Task 1 - Load and Explore the Dataset

In [ ]:
print("=" * 60)
print("TASK 1: LOAD AND EXPLORE THE DATASET")
print("=" * 60)

(x_train_raw, y_train), (x_test_raw, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

print("Number of training samples :", len(x_train_raw))
print("Number of testing samples  :", len(x_test_raw))
print("Shape of training data     :", x_train_raw.shape)
print("Shape of testing data      :", x_test_raw.shape)
print("Training  -> Positive:", int(np.sum(y_train == 1)), "| Negative:", int(np.sum(y_train == 0)))
print("Testing   -> Positive:", int(np.sum(y_test == 1)), "| Negative:", int(np.sum(y_test == 0)))
print("Example review (word indices):", x_train_raw[0][:15], "...")

## Task 2 - Data Preprocessing (Padding / Truncation)

In [ ]:
print("\n" + "=" * 60)
print("TASK 2: DATA PREPROCESSING")
print("=" * 60)

# Vocabulary is already limited to 10,000 words by num_words above.
# Padding makes every review exactly 200 numbers long.
x_train = pad_sequences(x_train_raw, maxlen=MAX_LEN)
x_test = pad_sequences(x_test_raw, maxlen=MAX_LEN)

print("Final training data shape:", x_train.shape)
print("Final testing data shape :", x_test.shape)

## Task 3 - Build the RNN Classifier

In [ ]:
def build_model(units=64, two_layers=False):
    """Input -> Embedding -> SimpleRNN (x1 or x2) -> Dense(sigmoid) output."""
    model = Sequential()
    model.add(Input(shape=(MAX_LEN,)))
    model.add(Embedding(VOCAB_SIZE, EMBED_DIM))       # word index -> 128-d vector
    if two_layers:
        # return_sequences=True passes the output of EVERY time step to the next RNN
        model.add(SimpleRNN(units, return_sequences=True))
        model.add(SimpleRNN(units))
    else:
        model.add(SimpleRNN(units))                   # keeps only the last hidden state
    model.add(Dense(1, activation="sigmoid"))         # probability of Positive review

    # Task 4 compile settings
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model


print("\n" + "=" * 60)
print("TASK 3: MODEL SUMMARY")
print("=" * 60)
build_model(units=64).summary()

## Helper: train + evaluate one model (used by all tasks below)

In [ ]:
def train_and_evaluate(units=64, epochs=5, batch_size=64, two_layers=False, verbose=0):
    model = build_model(units, two_layers)
    start = time.time()
    history = model.fit(x_train, y_train, epochs=epochs, batch_size=batch_size,
                        validation_split=0.2, verbose=verbose)
    train_time = time.time() - start
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    return {
        "model": model,
        "history": history.history,
        "train_acc": history.history["accuracy"][-1],
        "val_acc": history.history["val_accuracy"][-1],
        "train_loss": history.history["loss"][-1],
        "val_loss": history.history["val_loss"][-1],
        "test_acc": test_acc,
        "test_loss": test_loss,
        "time": train_time,
    }

## Task 4 - Compile and Train the Model (baseline: 64 units, 5 epochs, batch 64)

In [ ]:
print("\n" + "=" * 60)
print("TASK 4: TRAINING THE BASELINE MODEL")
print("=" * 60)

baseline = train_and_evaluate(units=64, epochs=5, batch_size=64, verbose=1)
model = baseline["model"]          # this is the model used in the app (Task 8)
hist = baseline["history"]

print(f"\nTraining accuracy   : {baseline['train_acc']:.4f}")
print(f"Validation accuracy : {baseline['val_acc']:.4f}")
print(f"Training loss       : {baseline['train_loss']:.4f}")
print(f"Validation loss     : {baseline['val_loss']:.4f}")

## Task 5 - Evaluate the Model

In [ ]:
print("\n" + "=" * 60)
print("TASK 5: EVALUATION ON TEST DATA")
print("=" * 60)

print(f"Test loss     : {baseline['test_loss']:.4f}")
print(f"Test accuracy : {baseline['test_acc']:.4f}")

print(f"\nQ1. Final test accuracy = {baseline['test_acc'] * 100:.2f}%")
if baseline["test_acc"] < baseline["train_acc"]:
    print("Q2. Test accuracy is LOWER than training accuracy. The model has partly "
          "memorised the training reviews (overfitting), so it does slightly worse "
          "on reviews it has never seen.")
else:
    print("Q2. Test accuracy is HIGHER than or equal to training accuracy. Training "
          "accuracy is measured while weights are still changing, so it can lag "
          "behind a model evaluated at the end of an epoch.")

## Task 6 - Performance Visualization

In [ ]:
def plot_history(h):
    epochs = range(1, len(h["accuracy"]) + 1)

    plt.figure(figsize=(6, 4))
    plt.plot(epochs, h["accuracy"], "o-", label="Training Accuracy")
    plt.plot(epochs, h["val_accuracy"], "s-", label="Validation Accuracy")
    plt.title("Graph 1: Training vs Validation Accuracy")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.savefig("accuracy_graph.png", dpi=150, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(6, 4))
    plt.plot(epochs, h["loss"], "o-", label="Training Loss")
    plt.plot(epochs, h["val_loss"], "s-", label="Validation Loss")
    plt.title("Graph 2: Training vs Validation Loss")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.savefig("loss_graph.png", dpi=150, bbox_inches="tight")
    plt.show()


print("\n" + "=" * 60)
print("TASK 6: PERFORMANCE VISUALIZATION")
print("=" * 60)
plot_history(hist)

print("Interpretation:")
print("- Accuracy graph: if training accuracy keeps rising while validation accuracy "
      "stays flat or drops, the model is starting to overfit.")
print("- Loss graph: training loss should fall each epoch. If validation loss starts "
      "rising, the model is memorising instead of learning general patterns.")

## Task 7 - Hyperparameter Experimentation

In [ ]:
print("\n" + "=" * 60)
print("TASK 7: HYPERPARAMETER EXPERIMENTS (this takes a few minutes)")
print("=" * 60)

experiments = [
    # (name, label,          units, epochs, batch)
    ("A1", "RNN = 32",        32,    5,     64),
    ("A2", "RNN = 64",        64,    5,     64),
    ("A3", "RNN = 128",      128,    5,     64),
    ("B1", "Epochs = 3",      64,    3,     64),
    ("B2", "Epochs = 5",      64,    5,     64),
    ("B3", "Epochs = 10",     64,   10,     64),
    ("C1", "Batch = 32",      64,    5,     32),
    ("C2", "Batch = 64",      64,    5,     64),
    ("C3", "Batch = 128",     64,    5,    128),
]

cache = {(64, 5, 64): baseline}    # A2 = B2 = C2 = baseline, so train it only once
rows = []
for name, label, units, epochs, batch in experiments:
    key = (units, epochs, batch)
    if key not in cache:
        print(f"Running {name} ({label}) ...")
        cache[key] = train_and_evaluate(units, epochs, batch)
    r = cache[key]
    rows.append({"Experiment": name, "Parameter": label,
                 "Training Accuracy": round(r["train_acc"], 4),
                 "Validation Accuracy": round(r["val_acc"], 4),
                 "Test Accuracy": round(r["test_acc"], 4)})

results = pd.DataFrame(rows)
print("\nRESULTS TABLE")
print(results.to_string(index=False))
results.to_csv("experiment_results.csv", index=False)

best = results.loc[results["Test Accuracy"].idxmax()]
print(f"\nBest experiment: {best['Experiment']} ({best['Parameter']}) "
      f"with test accuracy {best['Test Accuracy']:.4f}")

## Task 8 - Text Prediction (the application)

In [ ]:
word_index = imdb.get_word_index()


def encode_review(text):
    """Convert plain English text into the same format the model was trained on."""
    words = re.findall(r"[a-z']+", text.lower())
    seq = [1]                                  # 1 = start-of-review token in IMDB
    for w in words:
        idx = word_index.get(w)
        if idx is None or idx + 3 >= VOCAB_SIZE:
            seq.append(2)                      # 2 = unknown word
        else:
            seq.append(idx + 3)                # IMDB indices are shifted by 3
    return pad_sequences([seq], maxlen=MAX_LEN)


def predict_sentiment(review, trained_model=None):
    """Return ('Positive'/'Negative', probability that the review is positive)."""
    trained_model = trained_model or model
    prob = float(trained_model.predict(encode_review(review), verbose=0)[0][0])
    return ("Positive" if prob >= 0.5 else "Negative"), prob


print("\n" + "=" * 60)
print("TASK 8: TEXT PREDICTION")
print("=" * 60)

test_reviews = [
    ("This movie was fantastic, the acting was brilliant and the story kept me hooked till the end.", "Positive"),
    ("Terrible film. The plot made no sense, the acting was awful and I wasted my time.", "Negative"),
    ("A wonderful and touching story with beautiful music. I loved every minute of it.", "Positive"),
    ("Boring and predictable. I almost fell asleep, the worst movie I have seen this year.", "Negative"),
    ("An excellent thriller with great performances and a surprising ending. Highly recommended.", "Positive"),
]

pred_rows = []
for i, (text, expected) in enumerate(test_reviews, 1):
    label, prob = predict_sentiment(text)
    pred_rows.append({"Review": f"Review {i}", "Expected Sentiment": expected,
                      "Predicted Sentiment": label, "Probability (Positive)": round(prob, 4)})
print(pd.DataFrame(pred_rows).to_string(index=False))

## Task 9 - Challenge Activity: Single-layer vs Two-layer RNN

In [ ]:
print("\n" + "=" * 60)
print("TASK 9: CHALLENGE - SINGLE vs TWO-LAYER RNN")
print("=" * 60)

two_layer = train_and_evaluate(units=64, epochs=5, batch_size=64, two_layers=True)
two_layer["model"].summary()

compare = pd.DataFrame([
    {"Model": "Single-layer RNN", "Train Acc": round(baseline["train_acc"], 4),
     "Test Acc": round(baseline["test_acc"], 4), "Training Time (s)": round(baseline["time"], 1)},
    {"Model": "Two-layer RNN", "Train Acc": round(two_layer["train_acc"], 4),
     "Test Acc": round(two_layer["test_acc"], 4), "Training Time (s)": round(two_layer["time"], 1)},
])
print(compare.to_string(index=False))

better = "Two-layer" if two_layer["test_acc"] > baseline["test_acc"] else "Single-layer"
slower = "Two-layer" if two_layer["time"] > baseline["time"] else "Single-layer"
print(f"\n1. Better test accuracy : {better} RNN")
print(f"2. Takes longer to train: {slower} RNN")
print("3. Adding another RNN layer does NOT always improve performance. It adds "
      "parameters and makes vanishing gradients and overfitting worse.")
print("4. return_sequences=True makes the RNN output the hidden state at every time "
      "step (shape: batch x 200 x units) instead of only the last one. The next RNN "
      "layer needs a full sequence as input, so the first layer must use it.")

## Discussion Questions

In [ ]:
discussion = {
    "1. What is an RNN and why is it suitable for text classification?":
        "An RNN processes data step by step and passes a hidden state forward, so it remembers earlier words. "
        "Text is a sequence where word order matters, which makes RNNs a natural fit.",
    "2. Purpose of the Embedding layer?":
        "It turns each word index into a dense 128-number vector that is learned during training, so similar words get similar vectors.",
    "3. Why is padding required?":
        "A batch must be a fixed-size array. Padding/truncating makes every review exactly 200 words long.",
    "4. Why sigmoid in the output layer?":
        "Sigmoid squashes the output into 0-1, which can be read as the probability of a Positive review.",
    "5. Why binary cross-entropy?":
        "It is the standard loss for two-class problems. It heavily penalises confident wrong predictions.",
    "6. What is the vanishing-gradient problem?":
        "During backpropagation through many time steps, gradients get multiplied repeatedly by small numbers and shrink towards zero, "
        "so the network cannot learn long-range dependencies.",
    "7. What happens when RNN units are increased?":
        "The model can learn more complex patterns, but it has more parameters, trains slower and can overfit more easily.",
    "8. What is overfitting?":
        "The model performs very well on training data but poorly on unseen data because it memorised instead of generalising.",
    "9. How can overfitting be reduced?":
        "Use dropout, early stopping, fewer units, more data, regularisation, or switch to LSTM/GRU with dropout.",
    "10. SimpleRNN vs LSTM vs GRU?":
        "SimpleRNN has one simple hidden state and suffers from vanishing gradients. LSTM adds a memory cell with input, forget and output gates "
        "to remember long-term information. GRU is a lighter version with two gates (update, reset): faster, often similar accuracy.",
}

print("\n" + "=" * 60)
print("DISCUSSION QUESTIONS")
print("=" * 60)
for q, a in discussion.items():
    print(f"\n{q}\n   -> {a}")

## Conclusion (auto-filled from your results; rewrite it in your own words)

In [ ]:
def acc_of(name):
    return float(results.loc[results["Experiment"] == name, "Test Accuracy"].iloc[0])


units_trend = "improved" if acc_of("A3") > acc_of("A1") else "did not clearly improve"
epochs_trend = "improved" if acc_of("B3") > acc_of("B1") else "did not clearly improve"

conclusion = (
    f"In this lab I built a SimpleRNN sentiment analyzer for IMDB reviews using an Embedding layer, "
    f"a SimpleRNN layer and a sigmoid output. The baseline model (64 units, 5 epochs, batch 64) reached "
    f"a test accuracy of {baseline['test_acc'] * 100:.2f}%. The best result came from experiment "
    f"{best['Experiment']} ({best['Parameter']}) with {best['Test Accuracy'] * 100:.2f}% test accuracy. "
    f"Increasing RNN units {units_trend} performance, and increasing epochs {epochs_trend} performance. "
    f"The gap between training and validation accuracy shows mild overfitting, and the two-layer model "
    f"was {'not ' if better == 'Single-layer' else ''}better than the single-layer one while taking "
    f"{'longer' if slower == 'Two-layer' else 'less time'} to train. The main limitation of SimpleRNN is the "
    f"vanishing-gradient problem, which makes it forget the start of long reviews. LSTM or GRU layers "
    f"use gates to keep important information over long sequences, so they would likely give higher "
    f"and more stable accuracy on this task."
)
print("\n" + "=" * 60)
print("CONCLUSION")
print("=" * 60)
print(conclusion)

## Live Application: type your own review (type 'quit' to stop)

In [ ]:
print("\n" + "=" * 60)
print("MOVIE REVIEW SENTIMENT ANALYZER")
print("=" * 60)
while True:
    user_review = input("\nEnter a movie review (or 'quit'): ").strip()
    if user_review.lower() == "quit":
        print("Goodbye!")
        break
    if not user_review:
        continue
    label, prob = predict_sentiment(user_review)
    confidence = prob if label == "Positive" else 1 - prob
    print(f"Sentiment: {label}  (confidence: {confidence * 100:.1f}%)")